In [ ]:
"""
=============================================================================
 PIPELINE V5 — DÉMONSTRATION SUR IMAGE INDIVIDUELLE
=============================================================================
 PFE — Trustworthy AI for Opportunistic Screening of Cardiovascular Risk
 Abdallah OUAHMANE — FSR Rabat — Encadrant : Pr. Mohamed EL HASSOUNI

 FORMULE DE FUSION
   risk_global = (0.15 × s_ctr + 0.40 × s_cong
                + 0.10 × s_calcif + 0.35 × prob_cardio) × 100

 FONDEMENT EXPÉRIMENTAL (Chapitre 6)
   Étude 1  — le retrait de chaque composante dégrade l'AUC ; la fusion
              surpasse tout indicateur isolé (meilleur : 0.7037)
   Étude 2  — M3 fournit le signal calcique le plus discriminant ; toutes
              les normalisations de M2 font moins bien que l'absence de
              signal
   Étude 2B — l'ajout de M2 en cinquième composante n'apporte rien
              (+0.0002) ; M2 reste un module de visualisation
   Étude 3  — les pondérations retenues coïncident avec l'optimum de la
              grille contrainte ; la performance est stable dans un large
              voisinage
   Étude 4  — le gating par la confiance n'améliore pas le score : la
              confiance de M4 est informative mais mal calibrée
              (ECE = 0.0754)
   Étude 5  — le Module 4 seul atteint 0.9094, écart non significatif avec
              V5 (p = 0.1358) ; le pipeline conserve l'auditabilité
   Étude 6  — surajustement estimé à +0.0007 ; AUC en validation
              indépendante 0.8909 ± 0.0170

 RÔLE DES MODULES
   M1 — segmentation cardiaque, contribue à hauteur de 15 % du score
   M2 — segmentation des calcifications, visualisation uniquement
   M3 — classification des calcifications, contribue à hauteur de 10 %
   M4 — classification du risque, contribue à hauteur de 75 %
        (40 % congestion, 35 % cardiomégalie)

 SEUILS DE CATÉGORISATION
   FAIBLE  : score < 25 %
   MODÉRÉ  : 25 % ≤ score < 35 %
   ÉLEVÉ   : score ≥ 35 %   (maximise l'indice de Youden)
   INDÉTERMINÉ : CTR hors de la plage de plausibilité anatomique
=============================================================================
"""

import os
import cv2
import torch
import matplotlib
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torchvision import transforms, models
from PIL import Image

plt.rcParams['font.family'] = 'DejaVu Sans'

# ============================================================
# CONFIGURATION
# ============================================================
MODELS_DIR = '/kaggle/input/datasets/abdallahouahmane/pfe-cardio-models'
CSV_PATH   = '/kaggle/input/datasets/rahulogoel/clahe-enhancement-on-chestx-ray14/Data_Entry.csv'
IMG_DIR    = '/kaggle/input/datasets/rahulogoel/clahe-enhancement-on-chestx-ray14/IMAGE/IMAGE'
IMG_EXT    = '/kaggle/input/datasets/abdallahouahmane/images'

UNET_CARDIO  = f'{MODELS_DIR}/unet_cardio_robust_v2_best.pth'
UNET_CALCIF  = f'{MODELS_DIR}/unet_calcif_best_kagg (2).pth'
CALCIF_MODEL = f'{MODELS_DIR}/calcif_model_final.pth'
RISK_MODEL   = f'{MODELS_DIR}/cardio_risk_v2_best (1).pth'

CATEGORIES_RISK   = ['Normal', 'Cardiomegaly', 'Edema']
CATEGORIES_CALCIF = ['Normal', 'Calcification', 'Aortic_enlargement']

# --- Pondérations de fusion (Études 3 et 6) ---
W_CTR    = 0.15
W_CONG   = 0.40
W_CALCIF = 0.10
W_CARDIO = 0.35

# --- Seuils de catégorisation ---
SEUIL_ELEVE  = 35
SEUIL_MODERE = 25

# --- Seuil de binarisation des masques de segmentation ---
SEUIL_MASQUE = 0.5

# --- Plage de plausibilité anatomique du CTR (§7.4.3) ---
CTR_MIN, CTR_MAX = 0.30, 0.75

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("=" * 68)
print("  PIPELINE V5 — DÉMONSTRATION")
print("=" * 68)
print(f"  Device       : {device}")
print(f"  Fusion       : {W_CTR} × s_ctr + {W_CONG} × s_cong")
print(f"                 + {W_CALCIF} × s_calcif + {W_CARDIO} × prob_cardio")
print(f"  Seuil ÉLEVÉ  : ≥ {SEUIL_ELEVE} %")
print(f"  Seuil MODÉRÉ : ≥ {SEUIL_MODERE} %")
print(f"  M2           : visualisation, hors fusion (Étude 2B)")

# ============================================================
# TRANSFORMS
# ============================================================
T512 = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])
T224 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])


# ============================================================
# ARCHITECTURES
# ============================================================
class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super(UNet, self).__init__()

        def double_conv(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c), nn.ReLU(inplace=True))

        self.enc1 = double_conv(in_channels, 64)
        self.enc2 = double_conv(64, 128)
        self.enc3 = double_conv(128, 256)
        self.enc4 = double_conv(256, 512)
        self.pool = nn.MaxPool2d(2, 2)
        self.bottleneck = double_conv(512, 1024)
        self.up4 = nn.ConvTranspose2d(1024, 512, 2, 2)
        self.dec4 = double_conv(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec3 = double_conv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec2 = double_conv(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec1 = double_conv(128, 64)
        self.final = nn.Conv2d(64, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([e4, self.up4(b)], dim=1))
        d3 = self.dec3(torch.cat([e3, self.up3(d4)], dim=1))
        d2 = self.dec2(torch.cat([e2, self.up2(d3)], dim=1))
        d1 = self.dec1(torch.cat([e1, self.up1(d2)], dim=1))
        return torch.sigmoid(self.final(d1))


class EfficientNetCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(EfficientNetCNN, self).__init__()
        self.backbone = models.efficientnet_b0(weights=None)
        orig = self.backbone.features[0][0]
        self.backbone.features[0][0] = nn.Conv2d(
            1, orig.out_channels, kernel_size=orig.kernel_size,
            stride=orig.stride, padding=orig.padding, bias=False)
        in_f = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(in_f, 256),
            nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, num_classes))

        # Hooks pour Grad-CAM : activations à l'aller, gradients au retour
        self._act = None
        self._grad = None
        couche = self.backbone.features[-1]
        couche.register_forward_hook(self._save_act)
        couche.register_full_backward_hook(self._save_grad)

    def _save_act(self, m, i, o):
        self._act = o

    def _save_grad(self, m, gi, go):
        self._grad = go[0]

    def forward(self, x):
        return self.backbone(x)

    def get_gradcam(self, inp, classe=None):
        """Grad-CAM au sens de Selvaraju et al. : les activations sont
        pondérées par la moyenne spatiale des GRADIENTS de la classe
        ciblée, et non par leur propre moyenne."""
        self.eval()
        inp = inp.clone().requires_grad_(True)
        self._act = self._grad = None
        out = self.backbone(inp)
        idx = out.argmax(dim=1).item() if classe is None else classe
        self.zero_grad()
        out[0, idx].backward(retain_graph=True)
        if self._act is None or self._grad is None:
            return None, out
        act = self._act.detach().cpu().numpy()[0]
        grad = self._grad.detach().cpu().numpy()[0]
        poids = grad.mean(axis=(1, 2))
        cam = np.sum(poids[:, None, None] * act, axis=0)
        cam = np.clip(cam, 0, None)
        if cam.max() > 0:
            cam /= cam.max()
        return cam, out


# ============================================================
# CHARGEMENT DES MODÈLES
# ============================================================
def charger_modeles():
    """Les quatre modèles sont chargés en mode strict : toute
    incompatibilité entre le fichier de poids et l'architecture lève une
    exception. Ce contrôle est indispensable — l'usage de strict=False a
    masqué durant plusieurs semaines le chargement d'un checkpoint erroné
    pour le Module 3 (voir §7.4.3)."""
    print("\nChargement des modèles")

    m1 = UNet().to(device)
    m1.load_state_dict(torch.load(UNET_CARDIO, map_location=device))
    m1.eval()
    print("  M1 U-Net cardiaque       : OK   (Dice 0.9580)")

    m2 = UNet().to(device)
    m2.load_state_dict(torch.load(UNET_CALCIF, map_location=device))
    m2.eval()
    print("  M2 U-Net calcifications  : OK   (Dice 0.784)")

    m3 = EfficientNetCNN(num_classes=3).to(device)
    m3.load_state_dict(torch.load(CALCIF_MODEL, map_location=device))
    m3.eval()
    print("  M3 CNN calcifications    : OK   (AUC 0.7037)")

    m4 = EfficientNetCNN(num_classes=3).to(device)
    m4.load_state_dict(torch.load(RISK_MODEL, map_location=device))
    m4.eval()
    print("  M4 CNN risque            : OK   (AUC 0.9094)")

    return m1, m2, m3, m4


# ============================================================
# CALCUL DU CTR
# ============================================================
def calculer_ctr(mask, spacing, position):
    """Le CTR est un rapport de deux longueurs converties par le même
    facteur d'échelle : il est invariant par changement d'échelle isotrope.
    La conversion en centimètres sert à reporter les mesures en unités
    cliniques, non à corriger le ratio (§7.4.1).

    Le diamètre thoracique n'est pas segmenté mais approximé à 75 % de la
    largeur de l'image — limitation documentée en §7.4.1."""
    h, w = mask.shape
    contours, _ = cv2.findContours(mask.astype(np.uint8),
                                   cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    heart_px = 0
    if contours:
        c = max(contours, key=cv2.contourArea)
        x, y, wb, hb = cv2.boundingRect(c)
        cx, cy = x + wb // 2, y + hb // 2
        # Le contour retenu doit être centré dans la zone anatomique
        if w * 0.20 < cx < w * 0.80 and h * 0.20 < cy < h * 0.80:
            heart_px = wb

    thorax_px = int(w * 0.75)
    heart_cm = (heart_px * spacing) / 10
    thorax_cm = (thorax_px * spacing) / 10
    ctr = heart_cm / thorax_cm if (thorax_cm > 0 and heart_px > 0) else 0.0

    # Correction de la magnification en incidence antéro-postérieure
    if heart_px > 0 and 'AP' in str(position).upper():
        ctr -= 0.05

    valide = (CTR_MIN <= ctr <= CTR_MAX)
    return ctr, heart_cm, thorax_cm, valide


def categoriser(score, ctr_valide):
    if not ctr_valide:
        return 'INDÉTERMINÉ', '#7D3C98'
    if score >= SEUIL_ELEVE:
        return 'ÉLEVÉ', '#C0392B'
    if score >= SEUIL_MODERE:
        return 'MODÉRÉ', '#F39C12'
    return 'FAIBLE', '#1E8449'


# ============================================================
# ANALYSE D'UNE IMAGE
# ============================================================
def analyser(img_path, m1, m2, m3, m4, df_meta=None, image_name=None):
    nom = image_name or os.path.basename(img_path)
    img_pil = Image.open(img_path).convert('L')
    img_np = np.array(img_pil.resize((512, 512)))

    # --- Métadonnées ---
    spacing, position = 0.143, 'PA'
    age = sexe = label = 'N/A'
    largeur = img_pil.size[0]
    avertissement = ''

    if df_meta is not None and nom in df_meta.index:
        r = df_meta.loc[nom]
        spacing = float(r.get('OriginalImagePixelSpacing[x', 0.143))
        position = str(r.get('View Position', 'PA'))
        age = r.get('Patient Age', 'N/A')
        sexe = r.get('Patient Gender', 'N/A')
        label = r.get('Finding Labels', 'N/A')
        largeur = float(r.get('OriginalImage[Width', largeur))
        spacing_c = spacing * (largeur / 512.0)
    else:
        spacing_c = 0.143
        avertissement = 'mesures en cm non calibrées (métadonnées absentes)'

    inp512 = T512(img_pil).unsqueeze(0).to(device)
    inp224 = T224(img_pil).unsqueeze(0).to(device)

    # --- M1 : segmentation cardiaque et CTR ---
    with torch.no_grad():
        mask_c = (m1(inp512).cpu().squeeze().numpy()
                  > SEUIL_MASQUE).astype(np.uint8)
    ctr, heart_cm, thorax_cm, ctr_valide = calculer_ctr(
        mask_c, spacing_c, position)
    cardio_flag = ctr > 0.50

    # --- M2 : surface calcifiée (visualisation, hors fusion) ---
    with torch.no_grad():
        mask_k = (m2(inp512).cpu().squeeze().numpy()
                  > SEUIL_MASQUE).astype(np.uint8)
    calcif_pct = float(mask_k.sum()) / mask_k.size * 100

    # --- M3 : classification des calcifications ---
    with torch.no_grad():
        prob3 = torch.softmax(m3(inp224), dim=1).cpu().numpy()[0]
    calcif_label = CATEGORIES_CALCIF[int(np.argmax(prob3))]
    calcif_score = float(prob3[1]) + float(prob3[2])
    confiance_m3 = float(prob3.max())

    # --- M4 : classification du risque et Grad-CAM ---
    cam, out4 = m4.get_gradcam(inp224)
    prob4 = torch.softmax(out4, dim=1).detach().cpu().numpy()[0]
    risk_label = CATEGORIES_RISK[int(np.argmax(prob4))]
    confiance_m4 = float(prob4.max())

    # --- Scores normalisés ---
    s_ctr       = min(max((ctr - 0.40) / 0.20, 0.0), 1.0)
    s_calcif    = min(calcif_score, 1.0)
    s_cong      = float(prob4[2])
    prob_cardio = float(prob4[1])

    # --- Fusion V5 ---
    contributions = {
        'CTR':          W_CTR * s_ctr * 100,
        'Congestion':   W_CONG * s_cong * 100,
        'Calcique':     W_CALCIF * s_calcif * 100,
        'Cardiomégalie': W_CARDIO * prob_cardio * 100,
    }
    risk_global = round(sum(contributions.values()), 1)
    niveau, couleur = categoriser(risk_global, ctr_valide)

    # ------------------------------------------------------------------
    print("\n" + "=" * 68)
    print(f"  RAPPORT CARDIOVASCULAIRE — PIPELINE V5")
    print(f"  Image   : {nom}")
    print(f"  Label   : {label}")
    print(f"  Patient : âge {age} · sexe {sexe} · vue {position}")
    print("=" * 68)

    print(f"\n  MODULE 1 — TAILLE CARDIAQUE  (poids {W_CTR * 100:.0f} %)")
    print(f"    CTR       : {ctr:.3f}   "
          f"{'cardiomégalie' if cardio_flag else 'normal'}")
    print(f"    Cœur      : {heart_cm:.2f} cm | thorax : {thorax_cm:.2f} cm")
    if avertissement:
        print(f"                {avertissement}")
    if not ctr_valide:
        print(f"    ATTENTION : CTR hors de la plage de plausibilité "
              f"[{CTR_MIN} ; {CTR_MAX}] — résultat non exploitable")
    print(f"    s_ctr     : {s_ctr:.3f}  →  "
          f"{contributions['CTR']:.1f} / {W_CTR * 100:.0f} points")

    print(f"\n  MODULE 2 — SURFACE CALCIFIÉE  (visualisation, hors fusion)")
    print(f"    Surface segmentée : {calcif_pct:.2f} %")
    print(f"    Ce module ne contribue pas au score — voir Étude 2B")

    print(f"\n  MODULE 3 — CLASSIFICATION CALCIQUE  "
          f"(poids {W_CALCIF * 100:.0f} %)")
    for i, c in enumerate(CATEGORIES_CALCIF):
        print(f"    {c:<20} {prob3[i]:.3f}")
    print(f"    Diagnostic : {calcif_label} (confiance {confiance_m3:.3f})")
    print(f"    s_calcif   : {s_calcif:.3f}  →  "
          f"{contributions['Calcique']:.1f} / {W_CALCIF * 100:.0f} points")

    print(f"\n  MODULE 4 — RISQUE CARDIOVASCULAIRE  "
          f"(poids {(W_CONG + W_CARDIO) * 100:.0f} %)")
    for i, c in enumerate(CATEGORIES_RISK):
        print(f"    {c:<20} {prob4[i]:.3f}")
    print(f"    Diagnostic : {risk_label} (confiance {confiance_m4:.3f})")
    print(f"    s_cong      : {s_cong:.3f}  →  "
          f"{contributions['Congestion']:.1f} / {W_CONG * 100:.0f} points")
    print(f"    prob_cardio : {prob_cardio:.3f}  →  "
          f"{contributions['Cardiomégalie']:.1f} / {W_CARDIO * 100:.0f} points")

    print("\n" + "-" * 68)
    print(f"  FORMULE V5 : {W_CTR} × s_ctr + {W_CONG} × s_cong "
          f"+ {W_CALCIF} × s_calcif + {W_CARDIO} × prob_cardio")
    print(f"  RISQUE GLOBAL : {risk_global} %   [{niveau}]")
    for nom_c, val in contributions.items():
        print(f"    {nom_c:<16} {val:>5.1f}")
    print("=" * 68)

    # ------------------------------------------------------------------
    # VISUALISATION
    # ------------------------------------------------------------------
    fig = plt.figure(figsize=(22, 8))
    gs = gridspec.GridSpec(2, 4, figure=fig)

    ax = fig.add_subplot(gs[:, 0])
    ax.imshow(img_np, cmap='gray')
    ax.set_title('Radiographie', fontweight='bold')
    ax.axis('off')

    ax = fig.add_subplot(gs[0, 1])
    ax.imshow(img_np, cmap='gray')
    ax.imshow(np.ma.masked_where(mask_c == 0, mask_c),
              cmap=matplotlib.colors.ListedColormap(['#C0392B']), alpha=0.35)
    ax.set_title(f"M1 — CTR = {ctr:.3f}\ns_ctr = {s_ctr:.3f}",
                 fontweight='bold', fontsize=9,
                 color='#C0392B' if cardio_flag else '#1E8449')
    ax.axis('off')

    ax = fig.add_subplot(gs[0, 2])
    ax.imshow(img_np, cmap='gray')
    ax.imshow(np.ma.masked_where(mask_k == 0, mask_k),
              cmap='autumn', alpha=0.45)
    ax.set_title(f"M2 — {calcif_pct:.2f} % de surface\n(hors fusion)",
                 fontweight='bold', fontsize=9, color='#555555')
    ax.axis('off')

    ax = fig.add_subplot(gs[0, 3])
    if cam is not None:
        rgb = cv2.cvtColor(img_np, cv2.COLOR_GRAY2RGB)
        hm = cv2.applyColorMap(np.uint8(255 * cv2.resize(cam, (512, 512))),
                               cv2.COLORMAP_JET)
        ax.imshow(cv2.addWeighted(rgb, 0.5,
                                  cv2.cvtColor(hm, cv2.COLOR_BGR2RGB), 0.5, 0))
    else:
        ax.imshow(img_np, cmap='gray')
    ax.set_title(f"M4 — Grad-CAM\n{risk_label} (conf. {confiance_m4:.3f})",
                 fontweight='bold', fontsize=9)
    ax.axis('off')

    ax = fig.add_subplot(gs[1, 1])
    ax.barh(CATEGORIES_RISK, prob4 * 100,
            color=['#1E8449', '#C0392B', '#F39C12'])
    ax.set_xlim(0, 100)
    ax.set_xlabel('%')
    ax.set_title('M4 — probabilités', fontweight='bold', fontsize=9)
    for i, v in enumerate(prob4 * 100):
        ax.text(v + 1, i, f'{v:.1f}', va='center', fontsize=8)

    ax = fig.add_subplot(gs[1, 2])
    ax.barh(CATEGORIES_CALCIF, prob3 * 100,
            color=['#A9CCE3', '#7FB3D5', '#5499C7'])
    ax.set_xlim(0, 100)
    ax.set_xlabel('%')
    ax.set_title('M3 — probabilités', fontweight='bold', fontsize=9)
    for i, v in enumerate(prob3 * 100):
        ax.text(v + 1, i, f'{v:.1f}', va='center', fontsize=8)

    ax = fig.add_subplot(gs[1, 3])
    libelles = [f"CTR ({W_CTR*100:.0f} %)",
                f"Congestion ({W_CONG*100:.0f} %)",
                f"Calcique ({W_CALCIF*100:.0f} %)",
                f"Cardiomég. ({W_CARDIO*100:.0f} %)"]
    valeurs = [round(contributions[k], 1) for k in
               ['CTR', 'Congestion', 'Calcique', 'Cardiomégalie']]
    barres = ax.barh(libelles, valeurs,
                     color=['#1D9E75', '#185FA5', '#B45309', '#8E44AD'])
    ax.set_xlim(0, max(W_CONG, W_CARDIO) * 100 + 5)
    ax.set_xlabel('Points')
    ax.set_title(f'Pipeline V5 : {risk_global} %\n[{niveau}]',
                 fontweight='bold', fontsize=10, color=couleur)
    for b, v in zip(barres, valeurs):
        ax.text(b.get_width() + 0.5, b.get_y() + b.get_height() / 2,
                str(v), va='center', fontsize=9)

    plt.suptitle(f'Pipeline V5 — {nom}   |   Label : {label}   |   '
                 f'Score : {risk_global} % [{niveau}]',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    sortie = f"v5_{nom.rsplit('.', 1)[0]}.png"
    for ext in ('png', 'svg'):
        plt.savefig(sortie.replace('.png', f'.{ext}'), dpi=300,
                    bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"  Figure : {sortie}")

    return dict(image=nom, label=label, ctr=round(ctr, 3),
                ctr_valide=ctr_valide, cardio_flag=cardio_flag,
                heart_cm=round(heart_cm, 2), thorax_cm=round(thorax_cm, 2),
                calcif_pct=round(calcif_pct, 2), calcif_label=calcif_label,
                confiance_m3=round(confiance_m3, 3),
                confiance_m4=round(confiance_m4, 3),
                s_ctr=round(s_ctr, 3), s_cong=round(s_cong, 3),
                s_calcif=round(s_calcif, 3),
                prob_cardio=round(prob_cardio, 3),
                risk_label=risk_label, risk_global=risk_global,
                niveau=niveau)


# ============================================================
# PROGRAMME PRINCIPAL
# ============================================================
if __name__ == '__main__':
    m1, m2, m3, m4 = charger_modeles()

    df_meta = pd.read_csv(CSV_PATH).set_index('Image Index')
    print(f"\nMétadonnées NIH : {len(df_meta)} entrées")

    images = [
        (f'{IMG_DIR}/00000001_000.png', df_meta, '00000001_000.png'),
        (f'{IMG_EXT}/im1-Chest-X-ray.jpg', None, 'im1-Chest-X-ray.jpg'),
        (f'{IMG_EXT}/im2.png', None, 'im2.png'),
        (f'{IMG_EXT}/im3.png', None, 'im3.png'),
    ]

    resultats = []
    for chemin, meta, nom in images:
        if not os.path.exists(chemin):
            print(f"\n  Image introuvable : {chemin}")
            continue
        resultats.append(analyser(chemin, m1, m2, m3, m4, meta, nom))

    if resultats:
        print("\n" + "=" * 84)
        print("  SYNTHÈSE")
        print("=" * 84)
        print(f"  {'Image':<24}{'CTR':>7}{'s_ctr':>8}{'s_cong':>8}"
              f"{'s_calc':>8}{'p_card':>8}{'Score':>9}   Niveau")
        print("  " + "-" * 80)
        for r in resultats:
            print(f"  {r['image']:<24}{r['ctr']:>7.3f}{r['s_ctr']:>8.3f}"
                  f"{r['s_cong']:>8.3f}{r['s_calcif']:>8.3f}"
                  f"{r['prob_cardio']:>8.3f}{r['risk_global']:>8.1f} %"
                  f"   {r['niveau']}")

        pd.DataFrame(resultats).to_csv('resultats_demonstration_v5.csv',
                                       index=False)
        print("\n  CSV : resultats_demonstration_v5.csv")

    print(f"\n  Formule V5 : {W_CTR} × s_ctr + {W_CONG} × s_cong "
          f"+ {W_CALCIF} × s_calcif + {W_CARDIO} × prob_cardio")
    print(f"  AUC sur les 501 images de validation : 0.8916 "
          f"[IC 95 % 0.8572 – 0.9229]")
    print(f"  AUC en validation indépendante : 0.8909 ± 0.0170 (Étude 6)")
    print(f"  Au seuil de {SEUIL_ELEVE} % : sensibilité 0.928, "
          f"spécificité 0.766, VPP 0.888")
